# RAG 기초 (한글 예제)

**RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 는 LLM에게 질문할 때
**관련 문서를 찾아서 함께 넣어 주는** 방법입니다.

LLM은 학습한 내용만 알고 있습니다. 회사 내부 규정, 최신 자료, 개인 문서처럼
학습하지 않은 내용은 알지 못하고, 모르면서 아는 것처럼 지어내기도 합니다(환각).
RAG는 이 문제를 **모델을 다시 학습시키지 않고** 해결합니다.

## 전체 흐름

```
[준비 단계] 문서 → ① 청킹 → ② 임베딩 → ③ 벡터 DB 저장
[질문 단계] 질문 → ④ 유사한 청크 검색 → ⑤ 질문 + 청크를 LLM에 전달 → 답변
```

이 노트북에서는 가상의 회사 **한빛테크의 사내 정보보안 가이드**를 문서로 사용합니다.
회사 내부 규정이라 LLM이 절대 알 수 없는 내용이므로, RAG의 효과를 분명하게 확인할 수 있습니다.

## Step 0. 준비

필요한 라이브러리를 불러오고 API 키를 읽습니다.

In [1]:
import os
import shutil

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
load_dotenv(override=True)

if os.getenv("OPENAI_API_KEY"):
    print("OpenAI API Key 확인 완료")
else:
    print("OPENAI_API_KEY 가 없습니다. .env 파일을 확인하세요.")

# OpenAI 크레딧이 없어서 임베딩·답변 모두 로컬 Ollama 로 처리합니다.
# 크레딧을 충전하면 아래 세 줄을 "gpt-4.1-mini" / "text-embedding-3-small" 로 되돌리고,
# OpenAIEmbeddings(...) / ChatOpenAI(...) 의 base_url, api_key 인자를 지우세요.
OLLAMA_URL = os.getenv("OLLAMA_URL")
CHAT_MODEL = "qwen2.5:3b"            # 답변을 생성하는 모델 (ollama pull qwen2.5:3b)
EMBEDDING_MODEL = "bge-m3"           # 문장을 벡터로 바꾸는 모델 (ollama pull bge-m3, 한국어 지원)
DB_NAME = "chroma_rag_basic"          # 벡터 DB를 저장할 폴더 이름

OpenAI API Key 확인 완료


## Step 1. 문서 준비

실제 프로젝트에서는 PDF, 워드, 웹페이지 등을 불러오지만,
여기서는 내용을 눈으로 확인할 수 있도록 문서를 직접 작성해서 사용합니다.

In [3]:
document = """
한빛테크 사내 정보보안 가이드 (2026년판)

1. 비밀번호 정책
사내 모든 시스템의 비밀번호는 최소 12자 이상이어야 하며, 영문 대소문자와 숫자, 특수문자를 각각 1개 이상 포함해야 합니다.
비밀번호는 90일마다 변경해야 하고, 최근에 사용한 비밀번호 3개는 다시 사용할 수 없습니다.
모든 임직원은 다단계 인증(MFA)을 반드시 설정해야 하며, 인증 앱은 회사가 지정한 앱만 사용합니다.

2. 이메일 보안
외부에서 온 메일의 첨부파일은 열기 전에 반드시 발신자를 확인해야 합니다.
피싱으로 의심되는 메일을 받으면 링크를 누르지 말고 보안팀(security@hanbit.example)으로 신고합니다.
신고는 메일을 받은 시점으로부터 24시간 이내에 해야 합니다.
회사 메일을 개인 메일로 자동 전달하는 설정은 금지되어 있습니다.

3. 재택근무 보안
재택근무 시에는 회사가 제공한 VPN에 반드시 접속해야 합니다.
카페나 공항 등 공용 와이파이에서는 사내 시스템에 접속할 수 없습니다.
업무용 노트북은 5분 이상 사용하지 않으면 화면이 자동으로 잠기도록 설정해야 합니다.
가족을 포함한 타인에게 업무용 노트북을 빌려주는 것은 금지됩니다.

4. 데이터 분류와 취급
사내 데이터는 공개, 내부, 기밀, 극비의 4개 등급으로 분류합니다.
기밀 등급 이상의 자료는 회사 외부로 반출할 수 없으며, 반출이 필요한 경우 팀장과 정보보호책임자의 승인을 모두 받아야 합니다.
USB 등 이동식 저장장치는 사전 승인된 장치만 사용할 수 있고, 저장 시 반드시 암호화해야 합니다.
극비 자료는 지정된 보안 회의실에서만 열람할 수 있습니다.

5. 보안 사고 대응
랜섬웨어 감염이 의심되면 즉시 네트워크 연결을 끊고 보안관제센터(내선 1234)로 연락합니다.
전원을 끄지 말고 화면 상태를 그대로 유지해야 합니다. 증거가 사라질 수 있기 때문입니다.
모든 보안 사고는 인지한 시점부터 30분 이내에 보고해야 합니다.
사고 보고를 지연하거나 숨긴 경우 별도의 징계 절차가 진행됩니다.

6. 보안 교육과 점검
전 임직원은 연 2회 정보보안 교육을 이수해야 합니다.
분기마다 모의 피싱 훈련이 실시되며, 훈련 메일의 링크를 클릭한 직원은 추가 교육 대상이 됩니다.
교육을 이수하지 않으면 사내 시스템 계정이 일시적으로 제한될 수 있습니다.

7. 외부 서비스와 생성형 AI 이용
업무 자료를 외부 클라우드나 생성형 AI 서비스에 입력하는 것은 원칙적으로 금지됩니다.
사용이 필요한 경우 정보보호팀에 사전 신청서를 제출하고 승인을 받아야 합니다.
승인된 서비스라도 고객 개인정보와 소스코드는 입력할 수 없습니다.

8. 사무실 출입과 물리 보안
사무실 출입은 사원증으로만 가능하며, 다른 사람의 사원증을 빌려 쓰는 것은 금지됩니다.
방문객은 1층 안내데스크에서 방문증을 받아야 하고, 담당 직원이 동행해야 합니다.
퇴근 시에는 책상 위에 서류를 두지 않는 클린데스크 원칙을 지켜야 합니다.

9. 계정 관리
퇴사자의 계정은 퇴사일 당일에 즉시 비활성화합니다.
부서 이동이 있으면 3일 이내에 기존 권한을 회수하고 새 권한을 부여합니다.
공용 계정은 사용할 수 없으며, 모든 계정은 사용자 1명에게만 발급됩니다.
"""

print(f"문서 길이: {len(document):,}자")

문서 길이: 1,567자


## Step 2. 청킹 (Chunking)

문서를 통째로 LLM에 넣으면 두 가지 문제가 생깁니다.

1. 문서가 길면 **컨텍스트 한계**를 넘습니다.
2. 질문과 관계없는 내용까지 들어가서 **비용이 늘고 답변 정확도가 떨어집니다.**

그래서 문서를 적당한 크기의 조각(**청크**)으로 자른 뒤, 질문과 관련된 청크만 골라서 넣습니다.

**`RecursiveCharacterTextSplitter`** 는 문단(`\n\n`) → 줄(`\n`) → 문장 → 단어 순서로
**가능한 한 의미 단위를 지키면서** 자릅니다.

| 옵션 | 의미 |
|---|---|
| `chunk_size` | 청크 하나의 최대 길이 (여기서는 글자 수) |
| `chunk_overlap` | 앞 청크의 끝부분을 얼마나 겹쳐서 가져올지 |

`chunk_overlap`이 필요한 이유는, 자르는 지점에 걸친 문장이 두 청크 어디에서도
온전히 읽히지 않는 상황을 막기 위해서입니다.

> 한글은 영어보다 같은 글자 수에 담기는 정보량이 많습니다.
> 영어 예제에서 흔히 쓰는 `chunk_size=1000`보다 작게 잡는 편이 좋습니다.
> 이 노트북에서는 검색 과정을 눈으로 보기 쉽도록 250자로 잘게 나눴습니다.

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,        # 청크 하나의 최대 글자 수
    chunk_overlap=50,      # 앞 청크와 겹치는 글자 수
    separators=["\n\n", "\n", ". ", " ", ""],   # 자를 때 우선순위
)

chunks = splitter.split_text(document)

print(f"청크 개수: {len(chunks)}개")
print(f"평균 길이: {sum(len(c) for c in chunks) / len(chunks):.0f}자")

청크 개수: 9개
평균 길이: 172자


In [6]:
# 첫 두 청크를 직접 확인해 봅니다.
for i, chunk in enumerate(chunks[:3]):
    print(f"===== 청크 {i} ({len(chunk)}자) =====")
    print(chunk)
    print()

===== 청크 0 (220자) =====
한빛테크 사내 정보보안 가이드 (2026년판)

1. 비밀번호 정책
사내 모든 시스템의 비밀번호는 최소 12자 이상이어야 하며, 영문 대소문자와 숫자, 특수문자를 각각 1개 이상 포함해야 합니다.
비밀번호는 90일마다 변경해야 하고, 최근에 사용한 비밀번호 3개는 다시 사용할 수 없습니다.
모든 임직원은 다단계 인증(MFA)을 반드시 설정해야 하며, 인증 앱은 회사가 지정한 앱만 사용합니다.

===== 청크 1 (190자) =====
2. 이메일 보안
외부에서 온 메일의 첨부파일은 열기 전에 반드시 발신자를 확인해야 합니다.
피싱으로 의심되는 메일을 받으면 링크를 누르지 말고 보안팀(security@hanbit.example)으로 신고합니다.
신고는 메일을 받은 시점으로부터 24시간 이내에 해야 합니다.
회사 메일을 개인 메일로 자동 전달하는 설정은 금지되어 있습니다.

===== 청크 2 (171자) =====
3. 재택근무 보안
재택근무 시에는 회사가 제공한 VPN에 반드시 접속해야 합니다.
카페나 공항 등 공용 와이파이에서는 사내 시스템에 접속할 수 없습니다.
업무용 노트북은 5분 이상 사용하지 않으면 화면이 자동으로 잠기도록 설정해야 합니다.
가족을 포함한 타인에게 업무용 노트북을 빌려주는 것은 금지됩니다.



## Step 3. 임베딩 (Embedding)

**임베딩**은 문장을 숫자 벡터로 바꾸는 것입니다.
의미가 비슷한 문장은 벡터 공간에서 가까운 위치에 놓입니다.
컴퓨터는 이 거리를 계산해서 "의미가 비슷하다"를 판단합니다.

키워드 검색과의 차이가 여기에 있습니다.
"암호" 라고 검색해도 "비밀번호" 가 적힌 문서를 찾을 수 있습니다.

In [7]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_URL,
    api_key="ollama",
    check_embedding_ctx_length=False,   # OpenAI 전용 토큰 분할을 끄기 (Ollama 는 문자열을 그대로 받음)
)

# 무료로 쓰려면 아래처럼 HuggingFace 한국어 모델을 사용할 수도 있습니다.
# (langchain-huggingface, sentence-transformers 설치 필요, 첫 실행 시 모델 다운로드)
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

vector = embeddings.embed_query("비밀번호는 몇 자 이상이어야 하나요?")
print(f"벡터 차원: {len(vector)}")
print(f"앞 5개 값: {vector[:5]}")

벡터 차원: 1536
앞 5개 값: [0.04693603515625, -0.023223876953125, 0.05126953125, -0.01306915283203125, 0.030517578125]


### 의미가 비슷하면 정말 가까울까?

세 문장을 벡터로 바꾼 뒤 **코사인 유사도**를 계산해 봅니다.
1에 가까울수록 의미가 비슷하다는 뜻입니다.

> 절대적인 숫자보다 **상대적인 크기**가 중요합니다.
> 0.5가 낮아 보여도, 관계없는 문장(0.2)보다 훨씬 크면 "가깝다"고 판단합니다.

In [8]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b)


base =  "비밀번호는 12자 이상이어야 합니다."
similar = "암호는 열두 글자보다 길어야 해요."
different = "점심 메뉴는 김치찌개입니다."

v0, v1, v2 = embeddings.embed_documents([base, similar, different])

print(f"'{base}'")
print(f"  vs '{similar}'  →  유사도 {cosine_similarity(v0, v1):.3f}")
print(f"  vs '{different}'  →  유사도 {cosine_similarity(v0, v2):.3f}")

'비밀번호는 12자 이상이어야 합니다.'
  vs '암호는 열두 글자보다 길어야 해요.'  →  유사도 0.523
  vs '점심 메뉴는 김치찌개입니다.'  →  유사도 0.207


## Step 4. 벡터 DB에 저장

청크와 임베딩 벡터를 **벡터 데이터베이스**에 저장합니다.
여기서는 파일로 저장되는 가벼운 DB인 **Chroma**를 사용합니다.

> ⚠️ **자주 하는 실수:** `from_texts()` 에는 **청크 리스트**를 넣어야 합니다.
> 원문 문자열(`document`)을 그대로 넣으면 파이썬이 문자열을 한 글자씩 순회하기 때문에,
> **글자 하나하나가 문서로 저장**됩니다. 이렇게 되면 검색 결과가 공백 몇 개만 나오는데,
> 오류가 나지 않아서 원인을 찾기 어렵습니다.

In [9]:
# 이전에 만든 DB가 있으면 지우고 새로 만듭니다. (실습 반복을 위해)
if os.path.exists(DB_NAME):
    shutil.rmtree(DB_NAME)

# 어느 청크에서 나온 답인지 추적할 수 있도록 번호를 붙여 둡니다.
metadatas = [{"chunk_id": i} for i in range(len(chunks))]

vectorstore = Chroma.from_texts(
    texts=chunks,           # ← 반드시 청크 리스트! document(문자열)를 넣으면 안 됩니다.
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory=DB_NAME,
)

print(f"벡터 DB에 저장된 문서 수: {vectorstore._collection.count()}개")
print(f"(청크 개수 {len(chunks)}개와 같아야 정상입니다)")

벡터 DB에 저장된 문서 수: 9개
(청크 개수 9개와 같아야 정상입니다)


## Step 5. 검색 (Retrieval)

이제 질문과 **의미가 가까운 청크**를 찾아옵니다.
`k` 는 몇 개의 청크를 가져올지 정하는 값입니다.

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

question = "랜섬웨어에 걸린 것 같으면 어떻게 해야 하나요?"
docs = retriever.invoke(question)

print(f"질문: {question}\n")
for doc in docs:
    print(f"----- 청크 {doc.metadata['chunk_id']} -----")
    print(doc.page_content[:200])
    print()

질문: 랜섬웨어에 걸린 것 같으면 어떻게 해야 하나요?

----- 청크 4 -----
5. 보안 사고 대응
랜섬웨어 감염이 의심되면 즉시 네트워크 연결을 끊고 보안관제센터(내선 1234)로 연락합니다.
전원을 끄지 말고 화면 상태를 그대로 유지해야 합니다. 증거가 사라질 수 있기 때문입니다.
모든 보안 사고는 인지한 시점부터 30분 이내에 보고해야 합니다.
사고 보고를 지연하거나 숨긴 경우 별도의 징계 절차가 진행됩니다.

----- 청크 0 -----
한빛테크 사내 정보보안 가이드 (2026년판)

1. 비밀번호 정책
사내 모든 시스템의 비밀번호는 최소 12자 이상이어야 하며, 영문 대소문자와 숫자, 특수문자를 각각 1개 이상 포함해야 합니다.
비밀번호는 90일마다 변경해야 하고, 최근에 사용한 비밀번호 3개는 다시 사용할 수 없습니다.
모든 임직원은 다단계 인증(MFA)을 반드시 설정해야 하며, 인증 

----- 청크 6 -----
7. 외부 서비스와 생성형 AI 이용
업무 자료를 외부 클라우드나 생성형 AI 서비스에 입력하는 것은 원칙적으로 금지됩니다.
사용이 필요한 경우 정보보호팀에 사전 신청서를 제출하고 승인을 받아야 합니다.
승인된 서비스라도 고객 개인정보와 소스코드는 입력할 수 없습니다.



질문에 "랜섬웨어"라는 단어가 들어 있긴 하지만, 단순한 키워드 일치가 아니라
**의미가 가까운 순서로** 청크를 가져옵니다.
아래처럼 문서에 없는 표현으로 물어봐도 관련 내용을 찾아옵니다.

> **1위가 항상 정답은 아닙니다.** 아래 첫 번째 질문에서는 원하는 "재택근무" 내용이 2위로 나옵니다.
> 검색은 확률적이라 한 개만 가져오면 놓칠 수 있습니다. 그래서 보통 `k`를 2~5로 두어
> 여러 개를 가져온 뒤, 판단은 LLM에게 맡깁니다.

In [11]:
for q in ["집에서 일할 때 지켜야 할 것", "암호를 얼마나 자주 바꿔야 하나요?"]:
    docs = retriever.invoke(q)
    print(f"질문: {q}")
    for rank, doc in enumerate(docs, 1):
        first_line = doc.page_content.strip().splitlines()[0]
        print(f"  {rank}위  [청크 {doc.metadata['chunk_id']}] {first_line[:40]}")
    print()

질문: 집에서 일할 때 지켜야 할 것
  1위  [청크 5] 6. 보안 교육과 점검
  2위  [청크 2] 3. 재택근무 보안
  3위  [청크 7] 8. 사무실 출입과 물리 보안

질문: 암호를 얼마나 자주 바꿔야 하나요?
  1위  [청크 0] 한빛테크 사내 정보보안 가이드 (2026년판)
  2위  [청크 3] 4. 데이터 분류와 취급
  3위  [청크 8] 9. 계정 관리



## Step 6. LLM에게 질문하기

찾아온 청크를 **context**로 만들어 system 프롬프트에 넣고, 질문과 함께 LLM에 보냅니다.
이것이 RAG의 마지막 단계입니다.

프롬프트에 **"모르면 모른다고 답하라"** 는 지시를 넣는 것이 중요합니다.
문서에 없는 내용을 지어내지 않도록 막는 장치입니다.

In [12]:
SYSTEM_PROMPT_TEMPLATE = """당신은 한빛테크의 사내 보안 규정을 안내하는 도우미입니다.

아래 [문서] 내용만 근거로 답변하세요.
[문서]에 없는 내용은 추측하지 말고 "사내 문서에서 찾을 수 없습니다"라고 답하세요.
답변은 한국어로, 간결하게 작성하세요.

[문서]
{context}
"""

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, base_url=OLLAMA_URL, api_key="ollama")


def ask_rag(question):
    docs = retriever.invoke(question)                      # ① 관련 청크 검색
    context = "\n\n".join(doc.page_content for doc in docs)  # ② 컨텍스트로 합치기
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    response = llm.invoke([                                 # ③ 질문과 함께 LLM 호출
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    return response.content

In [13]:
print(ask_rag("비밀번호는 몇 자 이상이어야 하나요?"))

비밀번호는 최소 12자 이상이어야 합니다.


In [14]:
print(ask_rag("랜섬웨어에 걸린 것 같으면 어떻게 해야 하나요?"))

랜섬웨어 감염이 의심되면 즉시 네트워크 연결을 끊고 보안관제센터(내선 1234)로 연락해야 합니다. 전원을 끄지 말고 화면 상태를 그대로 유지해야 하며, 모든 보안 사고는 인지한 시점부터 30분 이내에 보고해야 합니다.


## Step 7. RAG의 효과 확인하기

같은 질문을 **RAG 없이** LLM에게 직접 물어보고 비교합니다.
사내 규정은 학습 데이터에 없기 때문에, LLM 혼자서는 답할 수 없습니다.

In [15]:
question = "피싱 의심 메일은 어디로, 언제까지 신고해야 하나요?"

print("===== RAG 없이 (LLM에게 직접 질문) =====")
print(llm.invoke(question).content)
print()
print("===== RAG 사용 =====")
print(ask_rag(question))

===== RAG 없이 (LLM에게 직접 질문) =====
피싱 의심 메일은 한국인터넷진흥원(KISA)에서 운영하는 '피싱신고센터'에 신고할 수 있습니다.  
신고 방법은 다음과 같습니다:

1. **이메일 전송**: 의심되는 피싱 메일을 그대로 첨부하여 phishing@kisa.or.kr 로 발송  
2. **온라인 신고**: KISA 피싱신고센터 웹사이트(https://www.phishing.or.kr)에서 신고 접수  
3. **전화 신고**: 국번 없이 118 (한국인터넷진흥원 인터넷침해대응센터)

**신고 기한**는 별도로 정해져 있지 않으며, 피싱 피해를 예방하기 위해 의심되는 메일을 받는 즉시 빠르게 신고하는 것이 중요합니다.  

또한, 피싱 메일을 받았을 때는 링크 클릭이나 첨부파일 실행을 하지 말고, 발신자 정보도 꼼꼼히 확인하는 것이 좋습니다.

===== RAG 사용 =====
피싱 의심 메일은 보안팀(security@hanbit.example)으로 메일을 받은 시점부터 24시간 이내에 신고해야 합니다.


### 문서에 없는 질문을 하면?

프롬프트에 넣은 지시가 제대로 동작하는지 확인합니다.
문서에 없는 내용은 지어내지 않고 모른다고 답해야 합니다.

In [16]:
print(ask_rag("연차 휴가는 며칠인가요?"))

사내 문서에서 찾을 수 없습니다.


## Step 8. 출처와 함께 답변하기

실무에서는 **어느 문서에서 나온 답인지** 함께 보여 주는 것이 중요합니다.
사용자가 답변을 검증할 수 있고, 환각도 빨리 발견할 수 있습니다.

In [17]:
def ask_rag_with_source(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])

    print(response.content)
    print("\n--- 참고한 청크 ---")
    for doc in docs:
        first_line = doc.page_content.strip().splitlines()[0]
        print(f"  [청크 {doc.metadata['chunk_id']}] {first_line[:50]}...")


ask_rag_with_source("USB를 사용하려면 어떻게 해야 하나요?")

USB 등 이동식 저장장치는 사전 승인된 장치만 사용할 수 있으며, 저장 시 반드시 암호화해야 합니다.

--- 참고한 청크 ---
  [청크 2] 3. 재택근무 보안...
  [청크 3] 4. 데이터 분류와 취급...
  [청크 1] 2. 이메일 보안...


## Step 9. (선택) Gradio 챗봇으로 만들기

아래 셀을 실행하면 브라우저에서 대화형으로 질문할 수 있습니다.
`ChatInterface`는 `(message, history)` 두 개의 인자를 넘겨주므로 함수 형태를 맞춰 줍니다.

In [18]:
import gradio as gr


def chat(message, history):
    return ask_rag(message)


gr.ChatInterface(chat, title="한빛테크 보안 규정 도우미").launch(inbrowser=True)

d:\ai_edu_main\llm_edu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 정리

| 단계 | 하는 일 | 사용한 도구 |
|---|---|---|
| ① 청킹 | 문서를 의미 단위로 자르기 | `RecursiveCharacterTextSplitter` |
| ② 임베딩 | 문장을 벡터로 바꾸기 | `OpenAIEmbeddings` |
| ③ 저장 | 벡터를 DB에 보관 | `Chroma` |
| ④ 검색 | 질문과 가까운 청크 찾기 | `retriever.invoke()` |
| ⑤ 생성 | 청크를 넣어 LLM에 질문 | `ChatOpenAI` |

**기억하기**

- RAG는 모델을 다시 학습시키지 않고 **모르는 지식을 넣어 주는** 방법입니다.
- 답변 품질은 **검색 품질**이 좌우합니다. 엉뚱한 청크를 가져오면 LLM도 엉뚱하게 답합니다.
- 프롬프트에 "문서에 없으면 모른다고 답하라"를 넣어 환각을 줄입니다.
- 출처를 함께 보여 주면 사용자가 답변을 검증할 수 있습니다.

## 연습 문제

1. `chunk_size`를 100으로 줄이고 400일 때와 답변을 비교해 보세요. 문장이 잘려서 답변이 어떻게 달라지나요?
2. `chunk_overlap`을 0으로 바꾸면 어떤 문제가 생기는지 확인해 보세요.
3. `k`를 1과 5로 바꿔 보세요. 적게 가져올 때와 많이 가져올 때 각각 어떤 장단점이 있나요?
4. `document` 내용을 여러분의 보유하고 있는 문서를 가지고 실행해 보세요